# Separation between levels (H3): graphs and embeddings

For each dimension and each pair of **instructed** levels, how distinct are the transcripts of the two levels?

* $\Delta$ = mean distance between transcripts of the two levels − mean distance between transcripts of the same level (pairs within one of the two levels). $\Delta > 0$ means the levels are separated.
* $p$: one-sided permutation test. The level labels of the transcripts of the two levels are shuffled (10,000 times) and $p$ is the share of shuffles giving a $\Delta$ at least as large. $p_\text{Holm}$ corrects for the 3 pairs of a dimension.

The test is run with the FGW distance between graphs and with the distance between embeddings of the whole transcript. The human column of the table (H1 contrast on perceived scores) comes from the perception analysis.

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cdist
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, holm, load_transcripts, separation
from graph_multimodalsentence import Graph
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
OUTPUT_DIR = "results"

DIMENSIONS = ["ability", "benevolence"]
PAIRS = [("Low", "Medium"), ("Medium", "High"), ("Low", "High")]
N_PER_LEVEL = 100               # transcripts per level and dimension (all pairs of them are compared)
SEED = 0
# Optional csv with a column id (<condition>#<row>): use only these transcripts, e.g. the
# videos of the perceptual study, so that graphs, embeddings and humans see the same stimuli.
TRANSCRIPT_IDS_FILE = None

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
EMBEDDING_METRIC = "cosine"     # same distance as in embedding_baseline.ipynb
N_PERMUTATIONS = 10000

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Transcripts

In [ ]:
df = load_transcripts(DATA_DIR, DIMENSIONS, reference="instructed")
if TRANSCRIPT_IDS_FILE:
    df = df[df["id"].isin(pd.read_csv(TRANSCRIPT_IDS_FILE)["id"])]
if N_PER_LEVEL is not None:
    df = pd.concat([group.sample(min(len(group), N_PER_LEVEL), random_state=SEED)
                    for _, group in df.groupby(["dimension", "level"])])
df = df.reset_index(drop=True)
df.groupby(["dimension", "level"]).size().unstack()[LEVELS]

## Distance matrices

All pairs of transcripts of a dimension. FGW matrices are cached in `results/`.

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


model = SentenceTransformer(EMBEDDING_MODEL)
embedding = CachedEncoder(model)
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")


def fgw_matrix(data, dimension):
    path = os.path.join(OUTPUT_DIR, f"h3_fgw_distances_{dimension}.npz")
    if os.path.exists(path):
        cached = np.load(path, allow_pickle=True)
        if list(cached["ids"]) == list(data["id"]) and cached["alpha"] == ALPHA:
            return cached["D"]
    start = time.time()
    graphs = [Graph.from_sentence(text, embedding) for text in data["text"]]
    n = len(graphs)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = fgw.graph_d(graphs[i], graphs[j])
    np.savez(path, D=D, ids=np.array(data["id"], dtype=object), alpha=ALPHA)
    print(f"{dimension}: {n * (n - 1) // 2} FGW distances in {time.time() - start:.0f} s")
    return D


def embedding_matrix(data):
    X = model.encode(data["text"].tolist(), batch_size=64)
    return cdist(X, X, metric=EMBEDDING_METRIC)


distances = {}
for dimension in DIMENSIONS:
    data = df[df["dimension"] == dimension]
    distances[(dimension, "Graph")] = fgw_matrix(data, dimension)
    distances[(dimension, "Emb.")] = embedding_matrix(data)

## Mean distance between and within levels

In [ ]:
def mean_distance(D, labels, a, b):
    # Mean distance between transcripts of levels a and b (distinct pairs when a == b).
    block = D[np.ix_(labels == a, labels == b)]
    return block[np.triu_indices(len(block), 1)].mean() if a == b else block.mean()


fig, axes = plt.subplots(len(DIMENSIONS), 2, figsize=(10, 4.2 * len(DIMENSIONS)), squeeze=False)
for row, dimension in enumerate(DIMENSIONS):
    labels = df.loc[df["dimension"] == dimension, "level"].to_numpy()
    for col, method in enumerate(["Graph", "Emb."]):
        D = distances[(dimension, method)]
        means = np.array([[mean_distance(D, labels, a, b) for b in LEVELS] for a in LEVELS])
        sns.heatmap(means, annot=True, fmt=".3g", cmap="viridis", xticklabels=LEVELS, yticklabels=LEVELS,
                    ax=axes[row, col], cbar=False)
        axes[row, col].set_title(f"{dimension}, {method} distance")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "h3_mean_distances.png"), dpi=300)
plt.show()

## Permutation tests

In [ ]:
rows = []
for dimension in DIMENSIONS:
    labels = df.loc[df["dimension"] == dimension, "level"].to_numpy()
    for method in ["Graph", "Emb."]:
        tests = [separation(distances[(dimension, method)], labels, a, b, N_PERMUTATIONS, SEED) for a, b in PAIRS]
        adjusted = holm([p for _, p, _, _ in tests])
        for (a, b), (delta, p, n_a, n_b), p_holm in zip(PAIRS, tests, adjusted):
            rows.append({"dimension": dimension, "pair": f"{a}--{b}", "method": method, "delta": delta,
                         "p": p, "p_holm": p_holm, "n": n_a + n_b})

separation_results = pd.DataFrame(rows)
separation_results.to_csv(os.path.join(OUTPUT_DIR, "h3_separation.csv"), index=False)
separation_results.pivot_table(index=["dimension", "pair"], columns="method",
                               values=["delta", "p", "p_holm"], sort=False).round(4)

## LaTeX rows

To paste into `tab:res:h3`. `P_COLUMN` chooses the reported $p$ (`"p_holm"` to match the Holm-corrected human column, then update the table note). The human and "Same?" columns stay `\tbd{}`.

In [ ]:
P_COLUMN = "p_holm"


def format_p(p):
    return "$<.001$" if p < 0.001 else f"{p:.3f}".lstrip("0")


def format_cell(row):
    return f"{row['delta']:.3g} ({format_p(row[P_COLUMN])})"


lines = []
for dimension in DIMENSIONS:
    lines.append(rf"\multicolumn{{5}}{{@{{}}l}}{{\textit{{{dimension.capitalize()}}}}} \\")
    for a, b in PAIRS:
        cells = separation_results[(separation_results["dimension"] == dimension)
                                   & (separation_results["pair"] == f"{a}--{b}")].set_index("method")
        pair = f"{a}--{b}".ljust(12)
        lines.append(rf"{pair} & {format_cell(cells.loc['Graph'])} & {format_cell(cells.loc['Emb.'])} & \tbd{{}} & \tbd{{}} \\")
print("\n".join(lines))